# imports

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import sys
import sklearn

sys.path.append('../src')
from load_clean_data import load_clean_data

In [2]:
df = load_clean_data()
print(df.shape)

c:\Users\corbi\Documents\Projects\sales-forecasting\sales-forecasting-project\notebooks\../src\load_clean_data.py:6: DtypeWarning: Columns (7: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  train_data = pd.read_csv('../data/train.csv')


(1017209, 18)


# Helpers

In [3]:
def evaluate_forecast(test_series, forecast):
    RMSE = sklearn.metrics.root_mean_squared_error(test_series, forecast)
    MAE = sklearn.metrics.mean_absolute_error(test_series, forecast)
    WAPE = 100 * (abs(test_series - forecast).sum()) / (abs(test_series).sum())
    
    print(f"RMSE: {RMSE:.2f}")
    print(f"MAE: {MAE:.2f}")
    print(f"WAPE: {WAPE:.2f}%")

# building the train-test split

train : dates up to june 30th 2015 

test: dates after june 30th 2015 up to july 30th 2015

In [4]:
store1 = df[df['Store'] == 1].sort_values('Date')

train  = store1[store1['Date'] < '2015-06-30']
test   = store1[store1['Date'] >= '2015-06-30']

In [5]:
length_check = len(train) + len(test) == len(store1)
order_check  = store1['Date'].is_monotonic_increasing

print(f"Length check: {length_check}, Order check: {order_check}")

Length check: True, Order check: True


In [6]:
train_series = train.set_index('Date')['Sales']
test_series  = test.set_index('Date')['Sales']

train_series = train_series.asfreq('D')
test_series = test_series.asfreq('D')

print(train_series.index)
print(test_series.index)
print(train_series.isnull().sum())
print(test_series.isnull().sum())

DatetimeIndex(['2013-01-01', '2013-01-02', '2013-01-03', '2013-01-04',
               '2013-01-05', '2013-01-06', '2013-01-07', '2013-01-08',
               '2013-01-09', '2013-01-10',
               ...
               '2015-06-20', '2015-06-21', '2015-06-22', '2015-06-23',
               '2015-06-24', '2015-06-25', '2015-06-26', '2015-06-27',
               '2015-06-28', '2015-06-29'],
              dtype='datetime64[us]', name='Date', length=910, freq='D')
DatetimeIndex(['2015-06-30', '2015-07-01', '2015-07-02', '2015-07-03',
               '2015-07-04', '2015-07-05', '2015-07-06', '2015-07-07',
               '2015-07-08', '2015-07-09', '2015-07-10', '2015-07-11',
               '2015-07-12', '2015-07-13', '2015-07-14', '2015-07-15',
               '2015-07-16', '2015-07-17', '2015-07-18', '2015-07-19',
               '2015-07-20', '2015-07-21', '2015-07-22', '2015-07-23',
               '2015-07-24', '2015-07-25', '2015-07-26', '2015-07-27',
               '2015-07-28', '2015-07-29

# SARIMA (single store)

### import

In [7]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

### setup

In [8]:
sarima_model = SARIMAX(train_series, 
                       order=(1, 1, 1), 
                       seasonal_order=(1, 1, 1, 7))

sarima_results = sarima_model.fit()

In [9]:
sarima_forecast = sarima_results.forecast(steps=len(test_series)) #type: ignore

sarima_forecast = sarima_forecast.clip(lower=0)

length_check = len(sarima_forecast) == len(test_series)
print(f"Length check: {length_check}")

Length check: True


In [10]:
print(sarima_forecast.head(10))
print(test_series.head(10))

2015-06-30    4622.776106
2015-07-01    4353.276278
2015-07-02    3917.249703
2015-07-03    4358.899833
2015-07-04    4818.950265
2015-07-05       0.000000
2015-07-06    4724.848108
2015-07-07    4484.255874
2015-07-08    4285.548274
2015-07-09    3893.117526
Freq: D, Name: predicted_mean, dtype: float64
Date
2015-06-30    5735
2015-07-01    5223
2015-07-02    5558
2015-07-03    4665
2015-07-04    4797
2015-07-05       0
2015-07-06    4359
2015-07-07    3650
2015-07-08    3797
2015-07-09    3897
Freq: D, Name: Sales, dtype: int64


### eval

In [11]:
evaluate_forecast(test_series, sarima_forecast)

RMSE: 703.38
MAE: 559.96
WAPE: 14.11%


comparison against baseline models store 1 results:

| Model | RMSE | MAE | WAPE |
|---|---|---|---|
| Naive (lag-7) | 1230.52 | 1069.38 | 26.94% |
| Seasonal Avg | 703.07 | 567.88 | 14.31% |
| SARIMA | 703.38 | 559.96 | 14.11% |

SARIMA and Seasonal Avg are effectively tied. SARIMA is slightly better on MAE and WAPE, but Seasonal Avg is slightly better on RMSE. both gaps are too small to claim a winner.

caveats:
- the baselines use real recent sales from inside the test month, but SARIMA forecasts all 32 days blind, so the baselines had the easier task
- this is one store over 32 days, so gaps this small could flip on a different store or month
- SARIMA hasn't been tuned and I haven't tried SARIMAX yet

# ETS (single store)

### import

In [12]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

### setup

In [13]:
ETS_model = ExponentialSmoothing(train_series,
                                 trend = None,
                                 seasonal = 'add',
                                 seasonal_periods = 7)

ETS_results = ETS_model.fit()

In [14]:
ETS_forecast = ETS_results.forecast(steps=len(test_series)) #type: ignore

ETS_forecast = ETS_forecast.clip(lower=0)

length_check = len(ETS_forecast) == len(test_series)
print(f"Length check: {length_check}")

Length check: True


In [15]:
print(ETS_forecast.head(10))
print(test_series.head(10))

2015-06-30    4212.551914
2015-07-01    3990.242714
2015-07-02    3645.118021
2015-07-03    4137.169930
2015-07-04    4556.925597
2015-07-05       0.000000
2015-07-06    4498.739398
2015-07-07    4212.551914
2015-07-08    3990.242714
2015-07-09    3645.118021
Freq: D, dtype: float64
Date
2015-06-30    5735
2015-07-01    5223
2015-07-02    5558
2015-07-03    4665
2015-07-04    4797
2015-07-05       0
2015-07-06    4359
2015-07-07    3650
2015-07-08    3797
2015-07-09    3897
Freq: D, Name: Sales, dtype: int64


### Eval

In [16]:
evaluate_forecast(test_series, ETS_forecast)

RMSE: 793.03
MAE: 608.60
WAPE: 15.33%


new results table:

| Model | RMSE | MAE | WAPE |
|---|---|---|---|
| Naive (lag-7) | 1230.52 | 1069.38 | 26.94% |
| Seasonal Avg | 703.07 | 567.88 | 14.31% |
| SARIMA | 703.38 | 559.96 | 14.11% |
| ETS | 793.03 | 608.60 | 15.33% |

# Prophet (Single Store)

### import

In [19]:
from prophet import Prophet

Importing plotly failed. Interactive plots will not work.


### making the ds / y table

In [17]:
prophet_ds_y_table = train[['Date', 'Sales']].rename(columns={'Date': 'ds', 'Sales': 'y'})

print(prophet_ds_y_table.head())
print(prophet_ds_y_table.shape[0])

                ds     y
1016095 2013-01-01     0
1014980 2013-01-02  5530
1013865 2013-01-03  4327
1012750 2013-01-04  4486
1011635 2013-01-05  4997
910


### setup

In [20]:
prophet_model = Prophet()

prophet_model_fitted = prophet_model.fit(prophet_ds_y_table)

16:42:05 - cmdstanpy - INFO - Chain [1] start processing
16:42:05 - cmdstanpy - INFO - Chain [1] done processing


In [24]:
prophet_future_df = prophet_model_fitted.make_future_dataframe(periods=len(test_series), freq='D')

print(prophet_future_df.head())
print(prophet_future_df.tail())
print(prophet_future_df.shape[0])

          ds
0 2013-01-01
1 2013-01-02
2 2013-01-03
3 2013-01-04
4 2013-01-05
            ds
937 2015-07-27
938 2015-07-28
939 2015-07-29
940 2015-07-30
941 2015-07-31
942


In [26]:
prophet_preds = prophet_model_fitted.predict(prophet_future_df)

prophet_preds = prophet_preds.tail(len(test_series))

prophet_forecast = prophet_preds.set_index('ds')['yhat'].clip(lower=0)

In [27]:
print(prophet_forecast.head(10))
print(test_series.head(10))

ds
2015-06-30    4219.709178
2015-07-01    4030.816088
2015-07-02    3662.776560
2015-07-03    4117.079554
2015-07-04    4578.022782
2015-07-05       0.000000
2015-07-06    4598.669638
2015-07-07    4323.929083
2015-07-08    4151.521833
2015-07-09    3797.890820
Name: yhat, dtype: float64
Date
2015-06-30    5735
2015-07-01    5223
2015-07-02    5558
2015-07-03    4665
2015-07-04    4797
2015-07-05       0
2015-07-06    4359
2015-07-07    3650
2015-07-08    3797
2015-07-09    3897
Freq: D, Name: Sales, dtype: int64


### eval

In [28]:
evaluate_forecast(test_series, prophet_forecast)

RMSE: 766.94
MAE: 607.52
WAPE: 15.31%


new results table:

| Model | RMSE | MAE | WAPE |
|---|---|---|---|
| Naive (lag-7) | 1230.52 | 1069.38 | 26.94% |
| Seasonal Avg | 703.07 | 567.88 | 14.31% |
| SARIMA | 703.38 | 559.96 | 14.11% |
| ETS | 793.03 | 608.60 | 15.33% |
| Prophet | 766.94 | 607.52 | 15.31% |